<a href="https://colab.research.google.com/github/gluekat/ByrdStuff/blob/master/Haunt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#import moviepy.editor as mpe
#mpe.config.IMAGEMAGICK_BINARY = '/usr/bin/convert'  # Or the correct path to your ImageMagick binary


#!pip install moviepy # --upgrade # This command updates to the latest version

'''
import moviepy.editor as mpe

# Check the current version of moviepy
print(mpe.__version__) # Added print statement to check version
# The config might be under editor. or might not be required for newer versions.
mpe.config.IMAGEMAGICK_BINARY = '/bin/convert'  # or the path to your imagemagick binary

'''





#from moviepy import *
from moviepy import TextClip
from moviepy import ImageClip
from PIL import Image
import numpy as np
import random
import io # import the io module
import os

#mpe.config.IMAGEMAGICK_BINARY = 'usr/bin/convert'

import matplotlib.font_manager

font_paths = matplotlib.font_manager.findSystemFonts(fontpaths="/usr/share/fonts/opentype/", fontext='otf')

# Default to the first OTF font if available
fpath = font_paths[0] if font_paths else None

target_font_name = "Arial"  # Replace with your desired font name

for font_path in font_paths:
    font_name = matplotlib.font_manager.FontProperties(fname=font_path).get_name()
    if font_name == target_font_name:
        fpath = font_path  # Update fpath if the target font is found
        break

print(f"Font path: {fpath}")

'''

import matplotlib.font_manager

font_paths = matplotlib.font_manager.findSystemFonts(fontpaths=None, fontext='otf')

for font_path in font_paths:
    fpath = font_path
    print(font_path)

'''
'''
import matplotlib.font_manager

font_paths = matplotlib.font_manager.findSystemFonts(fontpaths=None, fontext='otf')

# Access the first font path:
first_font_path = font_paths[0]
print("first path: " + first_font_path)
fpath = first_font_path

# Access a specific font path by name:
font_name = "Arial" # Replace with your desired font name
for font_path in font_paths:
    if font_name in font_path:
      fpath = font_path
      print("new fpath: " + fpath)
      break
'''

'''
Image Loading and Preparation:
Loads a base image from Google Drive.
Resizes the image for video output.
Converts the image to a NumPy array for manipulation.
'''

# Alternative Upload method
#from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

# Upload the image to your Google Drive and update the path below
image = Image.open('/content/drive/MyDrive/base_image.PNG').convert("RGB")

# Load image and resize
frame = np.array(ImageClip(np.array(image)).resized(height=720, width=1280).get_frame(0))
#ImageClip(np.array(image)).resized(height=720, width=1280).get_frame(0)

# Instead of directly using .resize, apply it through .fx
#frame = np.array(ImageClip(np.array(image)).fx(vfx.resize, width=1280, height=720).get_frame(0))
fps = 10
duration = 15
total_frames = fps * duration

'''
VCR Effect Generation:
generate_vcr_effect_frame function applies various distortions and effects to the base image to create a glitchy, VCR-like look.
These effects include rolling scan lines, color shifting, flickering, and noise.
'''


def generate_vcr_effect_frame(base_frame, i):
    frame = base_frame.copy()
    if i % random.randint(3, 6) == 0:
        for y in range(0, frame.shape[0], random.randint(4, 12)):
            frame[y] = np.roll(frame[y], random.randint(-20, 20), axis=0)
        frame = np.roll(frame, random.randint(-3, 3), axis=0)
        r = np.roll(frame[:, :, 0], random.randint(-3, 3), axis=1)
        g = np.roll(frame[:, :, 1], random.randint(-3, 3), axis=1)
        b = frame[:, :, 2]
        frame = np.stack([r, g, b], axis=2)
        for y in range(0, frame.shape[0], 2):
            frame[y] = (frame[y] * 0.6).astype(np.uint8)


#flicker
# flicker_strength is calculated using a sine wave, creating a fluctuating value between 0.6 and 1.0.
#This value is then multiplied with the frame's pixel values to create the flicker.

#flicker_intensity = 0.5: Reduces the flicker to half its original intensity.
#flicker_intensity = 2.0: Doubles the flicker intensity.
#flicker_intensity = 0: Removes the flicker completely.

    flk_intensity = 0

    flicker_strength = 0.8 + (0.2 * flk_intensity) * np.sin(2 * np.pi * i / 12)
    frame = np.clip(frame * flicker_strength, 0, 255).astype(np.uint8)


    frame = frame.astype(float)
    avg = frame.mean(axis=2, keepdims=True)
    frame = np.clip(frame + (frame - avg) * 0.8, 0, 255).astype(np.uint8)

#distortion
# adds a wiggle and distorteds image with lines
#distortion is a NumPy array representing random noise.
#It's added to the bottom 72 rows of the frame to create the distortion effect

    dist_intensity = 2

    distortion = (np.random.rand(72, frame.shape[1], 3) * 255 * 0.2 * dist_intensity).astype(np.uint8)
    frame[-72:] = np.clip(frame[-72:] + distortion, 0, 255)


    # Text overlay logic
    if i % random.randint(10, 30) == 0: # Controls flicker frequency


        txt_clip = TextClip(text = "Run", font_size = 40, color='red', font = fpath,
                            # Specify position using align and method
                            text_align='right', vertical_align = 'bottom')#, method='caption')

        #txt_clip = txt_clip.set_position(("center", "center")) # Adjust position as needed
        #txt_clip = txt_clip.set_duration(0.2) # Controls flicker duration

        frame = CompositeVideoClip([ImageClip(frame), txt_clip]).get_frame(0)

        #frame = CompositeVideoClip([ImageClip(frame).set_duration(0.2), txt_clip.set_start(0).set_end(0.2)]).get_frame(0)

    return frame # Added return statement to return the modified frame


'''
Video Creation:
Creates a sequence of frames using the generate_vcr_effect_frame function.
Combines the frames into a video using ImageSequenceClip
'''
frames = [generate_vcr_effect_frame(frame, i) for i in range(total_frames)]
video = ImageSequenceClip(frames, fps=fps)

'''
Audio (Commented Out):
Contains commented-out code for generating white noise audio and adding it to the video.

def generate_white_noise_audio(duration, sample_rate=44100):
    noise = np.random.normal(0, 1, int(duration * sample_rate)).astype(np.float32)
    noise = (noise / np.max(np.abs(noise))) * 0.2
    return AudioClip(lambda t: np.interp(t, np.linspace(0, duration, len(noise)), noise), duration=duration, fps=sample_rate)

audio = generate_white_noise_audio(duration)
video = video.set_audio(audio)
'''

video.write_videofile("vcr_glitch_haunted.mp4", codec="libx264", audio_codec="aac")







Font path: None
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ValueError: Invalid font None, pillow failed to use it with error 'NoneType' object has no attribute 'read'

In [ ]:
!apt-get update
!apt-get install imagemagick -y

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [73.0 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,697 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Hit:13 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

!which convert

#!pip install moviepy --upgrade # This command updates to the latest version



from moviepy import *

#from moviepy.editor import * as mpe

#mpe.config.IMAGEMAGICK_BINARY = '/bin/convert'


In [ ]:
!pip install moviepy --upgrade # This command updates to the latest version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 59.2 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.1.0
    Uninstalling pillow-11.1.0:
      Successfully uninstalled pillow-11.1.0
  Attempting uninstall: moviepy
    Found existing installation: moviepy 1.0.3
    Uninstalling moviepy-1.0.3:
      Successfully uninstalled moviepy-1.0.3


In [ ]:
'''
import os
target_dir = '/content/my_packages'
os.makedirs(target_dir, exist_ok=True)  # Create the directory if it doesn't exist

#!pip install --target="{target_dir}" moviepy
#!pip install moviepy
!pip install --target="/content/my_packages" moviepy


import sys
sys.path.append(target_dir)
'''

import moviepy as mpe
print(mpe.__version__)  # Prints the installed version

mpe.config.IMAGEMAGICK_BINARY = '/bin/convert'


2.1.1
